In [8]:
from pathlib import Path
import sys

import pandas as pd

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import load_sales
from src.models import SeasonalDateRegressor, SklearnRegressorConfig
from src.pipelines.train import split_by_time
from src.evaluation import regression_metrics
from src.features import add_time_features, add_lag_features, add_rolling_features
from src.pipelines import (
    build_revenue_ratio_targets,
    fit_models_full,
    predict_submission_from_models,
    train_validate_models,
)

data_root = repo_root / "data" / "datathon-2026-round-1"

In [9]:
df = load_sales(data_root=data_root, parse_dates=True)
train_df, valid_df = split_by_time(df, date_col="date", valid_fraction=0.2)
model = SeasonalDateRegressor()

X_train = train_df[["date"]]
y_train = train_df["Revenue"]

model.fit(X_train, y_train)

y_pred = model.predict(valid_df[["date"]])
y_true = valid_df["Revenue"]
metrics = regression_metrics(y_true, y_pred)
metrics

{'mae': 1612717.42000342,
 'rmse': 1955063.9313240314,
 'mape': 64.38999532946687,
 'smape': 44.72168107237182,
 'r2': -0.37689417787286095}

In [10]:
# Feature table
df = load_sales(data_root=data_root, parse_dates=True)
df = add_time_features(df, date_col="date")
df = add_lag_features(df, target_col="Revenue", lags=[1, 7, 14], date_col="date")
df = add_rolling_features(df, target_col="Revenue", windows=[7, 14], date_col="date")
df = df.dropna().reset_index(drop=True)

feature_cols = [col for col in df.columns if col not in ["date", "Revenue", "COGS"]]
df["ratio"] = df["Revenue"] / df["COGS"].clip(lower=1e-6)
targets = ["Revenue", "ratio"]
model_config = SklearnRegressorConfig(model_type="random_forest")

# 80/20 validation
validation_result = train_validate_models(
    df=df,
    features=feature_cols,
    targets=targets,
    model_config=model_config,
    date_col="date",
    valid_fraction=0.2,
)

print("Revenue metrics:", validation_result["metrics"]["Revenue"])
print(
    "Reconstructed COGS metrics:",
    regression_metrics(
        validation_result["valid_df"]["COGS"],
        validation_result["predictions"]["Revenue"]
        / validation_result["predictions"]["ratio"].clip(lower=1e-6),
    ),
)

# Full-data retrain for submission forecasting
models = fit_models_full(
    df=df,
    features=feature_cols,
    targets=targets,
    model_config=model_config,
    use_numpy=True,
)

Revenue metrics: {'mae': 569855.2729209478, 'rmse': 826603.0791405356, 'mape': 22.49313539894494, 'smape': 19.889699266160555, 'r2': 0.753659971755023}
Reconstructed COGS metrics: {'mae': 523486.84337459266, 'rmse': 769347.0517953181, 'mape': 22.029994567865373, 'smape': 20.414901912941048, 'r2': 0.7181243827689499}


In [11]:
# Submission prediction from full-data models
submission = predict_submission_from_models(
    models=models,
    feature_cols=feature_cols,
    start_date="2023-01-01",
    end_date="2024-07-01",
    revenue_model_name="Revenue",
    ratio_model_name="ratio",
    lags=(1, 7, 14),
    windows=(7, 14),
    eps=1e-6,
)

submission.head(), submission.tail()

(         date       Revenue          COGS
 0  2023-01-01  9.494591e+05  9.308209e+05
 1  2023-01-02  1.331133e+06  1.166402e+06
 2  2023-01-03  1.525705e+06  1.292069e+06
 3  2023-01-04  1.592151e+06  1.340097e+06
 4  2023-01-05  1.702993e+06  1.430928e+06,
            date       Revenue          COGS
 543  2024-06-27  7.648801e+06  7.663359e+06
 544  2024-06-28  7.309838e+06  7.324128e+06
 545  2024-06-29  7.502751e+06  7.502679e+06
 546  2024-06-30  8.324441e+06  8.297494e+06
 547  2024-07-01  7.877506e+06  7.862140e+06)

In [12]:
import os
os.makedirs(repo_root / "submissions", exist_ok=True)
submission.to_csv(repo_root / "submissions" / "baseline_submission.csv", index=False)